# demo_coop

> Phase 12, Track A (`specs/multiagent/phase12/`): better instructions and channels for Claude Haiku 5.5 on the Phase 11 clue board, compared paired on seed with the saved Phase 11 runs. Variants: A1 tells agents to follow the coordinator, A2 has the coordinator pass its pooled clues with each suggestion, A3 shows who works on what.

Artifacts under `results/multiagent/coop/{quick,pilot,full}/` (gitignored): `runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `coopA.png`. The response cache and spend ledger are Phase 10's (`results/multiagent/llm/`), so one $150 API cap covers all live spend.

- **Quick** (normal tests): every variant with the fake client; outputs identical for 1 and 8 processes, pooled and grid by grid.
- **Pilot** and **main** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live; the main runs only after `TRACKA_PLAN` is frozen.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

from explRL.bench_grid import readCSV, writeCSV
from explRL.clue_runs import runClueAny
from explRL.coop_grid import TRACKA_PLAN, coopHash, trackAGrid, trackAPilotGrid, trackAQuick
from explRL.coop_report import figTrackA, logMetrics, trackASummaries, writeTrackA
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.train import demo_artifact_dir

DN = 'multiagent/coop'
WORKERS = min(8, os.cpu_count() or 1)
LLM_WORKERS = 32
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
P11 = demo_artifact_dir('full', 'multiagent/clue')


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def readLog(p):
    return [json.loads(x) for x in open(p)] if p.exists() else []


def run(mode, grids, spec, workers=None, study='coopA'):
    "Run a workload through runWorkload; write runs.csv, turns.jsonl, timing.jsonl, report.md, coopA.png, metrics.json."
    out = demo_artifact_dir(mode, DN)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    workers = workers or (LLM_WORKERS if spec.get('kind') == 'live' else WORKERS)
    t0 = time.time()
    res = runWorkload(grids, spec, workers, str(logs), run=runClueAny)
    rows, wall = res['rows'], time.time() - t0
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    p11 = readCSV(P11 / 'runs.csv') if (P11 / 'runs.csv').exists() else []
    m = TRACKA_PLAN['settings']['m']
    S = trackASummaries(rows, p11, logMetrics(readLog(out / 'turns.jsonl'), m), logMetrics(readLog(P11 / 'turns.jsonl'), m), study)
    meta = dict(mode=mode, planHash=coopHash(TRACKA_PLAN), commit=gitCommit(), model=HAIKU, cap=CAP, wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)),
                calls=int(sum(r['calls'] for r in rows)), spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    writeTrackA(out / 'report.md', S, meta)
    if S['effect']:
        figTrackA(out, S)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      pooled=res['pooled'], workers=workers, timing=timing, plan=TRACKA_PLAN),
                                                 indent=2, sort_keys=True, default=str) + '\n')
    if spec.get('kind') == 'live':             # keep the live run's metrics and timing; a cached rerun overwrites the others
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent "
          f"in all, {wall:.0f} s; stopped: {res['stopped']}")
    return rows, S, meta

## Quick: every variant with the fake client (pipeline check)

In [ ]:
t0 = time.time()
rows, S, meta = run('quick', [('quick', trackAQuick())], dict(kind='fake'), study='coopAPilot')
assert time.time() - t0 < 30
q = demo_artifact_dir('quick', DN)
for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md'):
    assert (q / f).stat().st_size > 0, f
ref = (q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()
for w, pooled in ((1, True), (8, False)):
    run('quick', [('quick', trackAQuick())], dict(kind='fake', pooled=pooled), workers=w, study='coopAPilot')
    assert ((q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()) == ref
recs = readLog(q / 'turns.jsonl')
assert all('Teammates working on:' in x['obs'] for x in recs if x['key'].startswith('coopAPilot|coopAPilot|A3'))
assert all('Teammates working on:' not in x['obs'] for x in recs if not x['key'].startswith('coopAPilot|coopAPilot|A3'))
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:800])

## Pilot (`--flags llm`, live, 480 calls, about $0.12)

Each (variant, channel) pair at N = 4, seed 1100, 30 turns. The new prompt lines must be used without confusion: invalid replies under 5% (AC-8); otherwise stop and report.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    prows, _, pmeta = run('pilot', [('pilot', trackAPilotGrid())], LIVE_SPEC, study='coopAPilot')
    invalid = sum(r['invalid'] for r in prows) / max(sum(r['calls'] for r in prows), 1)
    print(f'invalid rate {invalid:.3f} (AC-8: < 0.05)')
    for r in prows:
        print(r['level'], r['channel'], f"welfare {r['welfare']:.2f}, invalid {r['invalid']:.0f}, shares {r['shares']:.0f}")

## Main (`--cell-timeout 0 --flags llm`, live, 9 600 calls, about $2.3)

A1 and A2 on the coordinator channel, A3 on the blackboard and the coordinator, N 4 and 8, seeds 900-904, 40 turns; compared paired on seed with the Phase 11 rows in `results/multiagent/clue/full/`.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert TRACKA_PLAN['frozen'], 'freeze and commit TRACKA_PLAN before the main runs'
    rows, S, meta = run('full', [('coopA', trackAGrid())], LIVE_SPEC)
    print(open(demo_artifact_dir('full', DN) / 'report.md').read()[:3000])